# 08.08 — Framework Validation

Run common split, holdout, preprocessing-policy, and target-availability checks.

In [1]:
# Import libraries
from pathlib import Path
import sys

In [2]:
# Define the root directory of the project
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

CONFIG_PATH = PROJECT_ROOT / "configs" / "modeling_foundation.yaml"
CONFIG_PATH

WindowsPath('e:/jcuenca/OneDrive - GUSCanada/5toTerm/01_Capstone/DataLocal/ontario-electricity-peak-risk/configs/modeling_foundation.yaml')

In [3]:
# Import module to manage modeling configuration, feature datasets, and directories
from src.ontario_peak_risk.modeling.common import (
    load_modeling_config,
    load_feature_dataset,
    ensure_modeling_directories,
)

In [4]:
# Load the modeling configuration, feature dataset, and ensure necessary directories exist
CONFIG, _ = load_modeling_config(CONFIG_PATH)
REPORTS_DIR, DOCS_DIR, OUTPUTS_DIR = ensure_modeling_directories(CONFIG, PROJECT_ROOT)
feature_dataset = load_feature_dataset(CONFIG, PROJECT_ROOT)
feature_dataset.shape

(262944, 100)

In [5]:
# Import modules for building validation folds and holdout fold, as well as validating temporal folds, holdout policy, and preprocessing policy
from src.ontario_peak_risk.modeling.splits import (
    build_validation_folds,
    build_final_holdout_fold,
)
from src.ontario_peak_risk.modeling.validation import (
    validate_temporal_folds,
    validate_holdout_policy,
    validate_preprocessing_policy,
)

In [6]:
# Build validation folds and final holdout fold
validation_folds = build_validation_folds(CONFIG)
final_holdout = build_final_holdout_fold(CONFIG)
all_folds = [*validation_folds, final_holdout]

# Validate temporal folds using the feature dataset and the defined time and group columns from the configuration
temporal_validation = validate_temporal_folds(
    feature_dataset,
    all_folds,
    time_column=CONFIG["modeling"]["time"]["time_column"],
    group_column=CONFIG["modeling"]["time"]["group_column"],
)
# Display the results of the temporal validation
temporal_validation


,fold,role,check,status
0,fold_2023,validation,non_empty_train,PASS
1,fold_2023,validation,non_empty_evaluation,PASS
2,fold_2023,validation,train_precedes_evaluation,PASS
3,fold_2023,validation,same_fsa_coverage,PASS
4,fold_2023,validation,no_temporal_overlap,PASS
5,fold_2023,validation,train_origin_is_horizon_safe,PASS
6,fold_2023,validation,evaluation_origin_is_horizon_safe,PASS
7,fold_2024,validation,non_empty_train,PASS
8,fold_2024,validation,non_empty_evaluation,PASS
9,fold_2024,validation,train_precedes_evaluation,PASS


In [7]:
# Validate the holdout policy using the configuration
holdout_validation = validate_holdout_policy(CONFIG)
holdout_validation


,check,status,latest_validation_end,test_start
0,final_holdout_after_all_validation_periods,PASS,2024-12-31 23:00:00,2025-01-01


In [8]:
# Validate the preprocessing policy using the configuration
preprocessing_validation = validate_preprocessing_policy(CONFIG)
preprocessing_validation


,check,status,configured_value
0,preprocessing_fit_scope_training_only,PASS,training_only


In [9]:
# Assert that all validations passed
assert (temporal_validation["status"] == "PASS").all()
assert (holdout_validation["status"] == "PASS").all()
assert (preprocessing_validation["status"] == "PASS").all()

print("All common Modeling Foundation policy checks passed.")


All common Modeling Foundation policy checks passed.


In [10]:
# Run the complete Modeling Foundation validation
from src.ontario_peak_risk.modeling.run_modeling_foundation import (
    run_modeling_foundation,
)

results = run_modeling_foundation(
    CONFIG_PATH
)

Modeling Foundation Phase completed successfully.
Feature rows: 262,944
Complete Forecasting target rows: 262,800
Maximum forecast horizon: 24 hours
Forecasting selection metric: mae
Peak-Risk selection metric: pr_auc
Initial classification threshold: 0.5
Validation folds: 2
Final holdout: test_2025
Reports: E:\jcuenca\OneDrive - GUSCanada\5toTerm\01_Capstone\DataLocal\ontario-electricity-peak-risk\reports\modeling_foundation
Documentation: E:\jcuenca\OneDrive - GUSCanada\5toTerm\01_Capstone\DataLocal\ontario-electricity-peak-risk\docs\modeling_foundation


In [11]:
# Review horizon-safe temporal splits
display(
    results["split_summary"]
)

,fold,role,configured_train_start,configured_train_end,safe_train_origin_end,train_rows,train_first_origin,train_last_origin,train_fsas,configured_evaluation_start,configured_evaluation_end,safe_evaluation_origin_end,evaluation_rows,evaluation_first_origin,evaluation_last_origin,evaluation_fsas,max_horizon_hours
0,fold_2023,validation,2021-01-01,2022-12-31 23:00:00,2022-12-30 23:00:00,104976,2021-01-01,2022-12-30 23:00:00,6,2023-01-01,2023-12-31 23:00:00,2023-12-30 23:00:00,52416,2023-01-01,2023-12-30 23:00:00,6,24
1,fold_2024,validation,2021-01-01,2023-12-31 23:00:00,2023-12-30 23:00:00,157536,2021-01-01,2023-12-30 23:00:00,6,2024-01-01,2024-12-31 23:00:00,2024-12-30 23:00:00,52560,2024-01-01,2024-12-30 23:00:00,6,24
2,test_2025,test,2021-01-01,2024-12-31 23:00:00,2024-12-30 23:00:00,210240,2021-01-01,2024-12-30 23:00:00,6,2025-01-01,2025-12-31 23:00:00,2025-12-30 23:00:00,52416,2025-01-01,2025-12-30 23:00:00,6,24


In [12]:
# Review FSA coverage
display(
    results["fsa_coverage"]
)

,fold,role,fsa,train_rows,evaluation_rows,status
0,fold_2023,validation,L4T,17496,8736,PASS
1,fold_2023,validation,M5R,17496,8736,PASS
2,fold_2023,validation,M5S,17496,8736,PASS
3,fold_2023,validation,M6G,17496,8736,PASS
4,fold_2023,validation,M9R,17496,8736,PASS
5,fold_2023,validation,M9W,17496,8736,PASS
6,fold_2024,validation,L4T,26256,8760,PASS
7,fold_2024,validation,M5R,26256,8760,PASS
8,fold_2024,validation,M5S,26256,8760,PASS
9,fold_2024,validation,M6G,26256,8760,PASS


In [13]:
# Review Peak-Risk multi-horizon policy
display(
    results["peak_horizon_policy"]
)

,horizon,target,target_offset_hours,definition,threshold_scope
0,1,peak_h01,1,Peak status of the corresponding FSA at foreca...,training_only_fsa_season
1,2,peak_h02,2,Peak status of the corresponding FSA at foreca...,training_only_fsa_season
2,3,peak_h03,3,Peak status of the corresponding FSA at foreca...,training_only_fsa_season
3,4,peak_h04,4,Peak status of the corresponding FSA at foreca...,training_only_fsa_season
4,5,peak_h05,5,Peak status of the corresponding FSA at foreca...,training_only_fsa_season
5,6,peak_h06,6,Peak status of the corresponding FSA at foreca...,training_only_fsa_season
6,7,peak_h07,7,Peak status of the corresponding FSA at foreca...,training_only_fsa_season
7,8,peak_h08,8,Peak status of the corresponding FSA at foreca...,training_only_fsa_season
8,9,peak_h09,9,Peak status of the corresponding FSA at foreca...,training_only_fsa_season
9,10,peak_h10,10,Peak status of the corresponding FSA at foreca...,training_only_fsa_season


In [14]:
# Review all framework validation checks
display(
    results["framework_validation"]
)

,fold,role,check,status,details,latest_validation_end,test_start,configured_value
0,fold_2023,validation,non_empty_train,PASS,NaN,NaT,NaT,NaN
1,fold_2023,validation,non_empty_evaluation,PASS,NaN,NaT,NaT,NaN
2,fold_2023,validation,train_precedes_evaluation,PASS,NaN,NaT,NaT,NaN
3,fold_2023,validation,same_fsa_coverage,PASS,NaN,NaT,NaT,NaN
4,fold_2023,validation,no_temporal_overlap,PASS,NaN,NaT,NaT,NaN
5,fold_2023,validation,train_origin_is_horizon_safe,PASS,NaN,NaT,NaT,NaN
6,fold_2023,validation,evaluation_origin_is_horizon_safe,PASS,NaN,NaT,NaT,NaN
7,fold_2024,validation,non_empty_train,PASS,NaN,NaT,NaT,NaN
8,fold_2024,validation,non_empty_evaluation,PASS,NaN,NaT,NaT,NaN
9,fold_2024,validation,train_precedes_evaluation,PASS,NaN,NaT,NaT,NaN


In [15]:
# Final validation
validation_ok = (
    results[
        "framework_validation"
    ]["status"]
    .eq("PASS")
    .all()
)

fsa_coverage_ok = (
    results[
        "fsa_coverage"
    ]["status"]
    .eq("PASS")
    .all()
)

assert validation_ok, (
    "Framework validation contains FAIL checks."
)

assert fsa_coverage_ok, (
    "At least one FSA has insufficient fold coverage."
)

print(
    "Modeling Foundation is ready for final review."
)

Modeling Foundation is ready for final review.
